In [1]:
"""
PIPELINE STAGE: Renewable Electricity Generation Table Extraction from Monthly Energy Reports

INPUT: raw_data/energy_reports/{2021-2025}/*.docx
OUTPUT: processed_data/steps/04_Generation.xlsx
LIBRARIES: os, zipfile, xml.etree.ElementTree, pandas

1. OBJECTIVE:
    Extract provincial renewable electricity generation tables from monthly energy
    reports and consolidate them into a single master dataset. The workflow operates
    directly on DOCX XML structures, identifies renewable generation tables, enriches
    them with temporal metadata, and exports a unified dataset for energy analytics,
    environmental assessment, and machine learning applications.

2. DOCUMENT PARSING & XML EXTRACTION:
    A. Direct DOCX Access:
       - Treat DOCX files as ZIP archives.
       - Read the internal file:

           word/document.xml

       - Parse XML content using ElementTree.

    B. Word Table Extraction:
       - Locate all table elements (<w:tbl>).
       - Extract rows (<w:tr>) and cells (<w:tc>).
       - Merge text fragments (<w:t>) within each cell.
       - Exclude completely empty rows.

    C. Table Normalization:
       - Determine the maximum column count for each table.
       - Fill missing cells with empty strings.
       - Convert normalized structures into pandas DataFrames.
       - Ignore tables containing fewer than two rows.

3. TARGET TABLE DETECTION:
    A. Content-Based Identification:
       Analyze both:

           - Column headers
           - First data row

       after text normalization.

    B. Character Normalization:
       Apply:

           replace("i̇", "i")

       to eliminate Turkish character encoding inconsistencies.

    C. Required Renewable Keywords:
       A table is considered a renewable-energy table only if
       all of the following keywords are present:

           * iller
           * biyokütle
           * güneş
           * rüzgar

    D. Candidate Collection:
       - Store all matching tables.
       - Ignore unrelated report tables.

4. HEADER CORRECTION & STRUCTURAL REPAIR:
    A. Header Validation:
       Verify whether the extracted column names contain:

           "iller"

    B. Header Recovery:
       If the keyword is not found:

           - Promote the first data row to column headers.
           - Remove the promoted row.
           - Reset row indexing.

    C. Structural Preservation:
       Maintain the original renewable generation table layout.

5. TEMPORAL METADATA EXTRACTION:
    A. Filename Parsing:
       Extract month and year from filenames formatted as:

           january_2023.docx
           august_2024.docx
           december_2025.docx

    B. Month Formatting:
       - Read the month string directly.
       - Capitalize the first character.

       Examples:

           january → January
           july    → July

    C. Year Assignment:
       - Use the year value from the filename.
       - If unavailable, use the parent folder name.

6. GENERATION TABLE SELECTION LOGIC:
    A. Variable Report Structure:
       Monthly reports may contain:

           - Capacity table + Generation table
           - Only Generation table

    B. Robust Extraction Strategy:
       Instead of selecting a fixed table position,
       always select:

           The last matching renewable-energy table

           valid_tables[-1]

    C. Rationale:
       - In standard reports:
             1st table → Installed Capacity
             2nd table → Electricity Generation

       - In exceptional reports:
             Only 1 matching table exists
             and it represents electricity generation.

       Selecting the last matching table ensures
       consistent generation extraction across all reports.

7. SPECIAL CASE HANDLING:
    A. Single Table Reports:
       If only one matching table exists:

           - Treat it as the generation table.
           - Continue processing normally.

    B. Missing Table Reports:
       If no matching tables exist:

           - Skip the file.
           - Generate a warning message.

    C. Informational Logging:
       Report files containing only a single
       renewable-energy table.

8. TEMPORAL ENRICHMENT:
    Append the following variables to every extracted table:

           Year
           Month

    before adding the dataset to the master collection.

9. MULTI-YEAR CONSOLIDATION:
    A. Target Years:

           2021
           2022
           2023
           2024
           2025

    B. Directory Traversal:
       - Scan every yearly report folder.
       - Process all DOCX files.

    C. File Filtering:
       Ignore:

           - Temporary Word files (~*)
           - Non-DOCX files

    D. Dataset Assembly:
       Store all extracted monthly generation tables
       and merge them into a single master dataset.

10. OUTPUT GENERATION:
    A. Final Dataset Construction:
       Combine all extracted tables using:

           pd.concat(..., ignore_index=True)

    B. Export Destination:

           processed_data/steps/04_Generation.xlsx

    C. Export Settings:
       - Excel format (.xlsx)
       - No index column

11. PROCESS LOGGING & MONITORING:
    A. Runtime Messages:
       Display:

           - Current year
           - Processed file name
           - Extracted year
           - Extracted month

    B. Informational Messages:
       Report files containing only one valid
       renewable-energy table.

    C. Warning Messages:
       Report files containing no valid
       renewable-energy tables.

    D. Completion Messages:
       Confirm successful creation and save location
       of the generation master dataset.

12. EXPECTED OUTPUT DATASET:
    The resulting dataset contains:

        - Province-level renewable electricity generation values
        - Renewable energy source generation indicators
        - Year
        - Month

    The dataset represents the raw renewable electricity
    generation master table and serves as a primary input
    for cleaning, harmonization, integration with installed
    capacity data, emissions analysis, clustering studies,
    and machine learning workflows.

"""

import os
import zipfile
import xml.etree.ElementTree as ET
import pandas as pd

class GenerationExtractor:
    def __init__(self, base_dir):
        self.raw_root = os.path.join(base_dir, "raw_data", "energy_reports")
        self.final_dir = os.path.join(base_dir, "processed_data", "steps")
        self.ns = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}
        
        os.makedirs(self.final_dir, exist_ok=True)

    def _parse_xml_table(self, table_el):
        rows = []
        for row_el in table_el.findall('.//w:tr', self.ns):
            cells = []
            for cell_el in row_el.findall('.//w:tc', self.ns):
                cell_text = "".join([t.text for t in cell_el.findall('.//w:t', self.ns) if t.text])
                cells.append(cell_text.strip())
            
            if any(cells):
                rows.append(cells)
        
        if len(rows) < 2:
            return None
            
        max_cols = max(len(row) for row in rows)
        normalized_rows = [row + [''] * (max_cols - len(row)) for row in rows]
        return pd.DataFrame(normalized_rows[1:], columns=normalized_rows[0])

    def _is_target_table(self, df):
        if df is None or len(df) < 1:
            return False
            
        cols_text = " ".join([str(c) for c in df.columns]).lower().replace("i̇", "i")
        row0_text = " ".join([str(c) for c in df.iloc[0]]).lower().replace("i̇", "i")
        combined = cols_text + " " + row0_text
        required_words = ["iller", "biyokütle", "güneş", "rüzgar"]

        
        return all(w in combined for w in required_words)

    def extract_tables(self, file_path):
        valid_tables = []
        with zipfile.ZipFile(file_path, 'r') as docx_zip:
            xml_content = docx_zip.read('word/document.xml')
            root = ET.fromstring(xml_content)
            
            for table_el in root.findall('.//w:tbl', self.ns):
                df = self._parse_xml_table(table_el)
                
                if self._is_target_table(df):
                    cols_lower = " ".join([str(c) for c in df.columns]).lower().replace("i̇", "i")
                    if "iller" not in cols_lower:
                        df.columns = df.iloc[0]
                        df = df.drop(0).reset_index(drop=True)
                    valid_tables.append(df)
                    
        return valid_tables

    def run_for_file(self, year_folder, docx_name, master_generation_list):
        file_path = os.path.join(self.raw_root, year_folder, docx_name)
        if not os.path.exists(file_path):
            return

        base_name = os.path.splitext(docx_name)[0]
        parts = base_name.split('_')
        
        # Ayı doğrudan isim olarak al ve baş harfini büyüt (Örn: january -> January)
        month_val = parts[0].capitalize() if len(parts) > 0 else ""
        year_val = parts[1] if len(parts) > 1 else year_folder

        print(f">>> Hafızaya Alınıyor (Generation): {year_folder}/{docx_name} -> (Yıl: {year_val}, Ay: {month_val})")
        valid_tables = self.extract_tables(file_path)
        
        # Eşleşen tablo bulunduysa
        if len(valid_tables) > 0:
            # Genelde 1. tablo Kurulu Güç, 2. tablo Üretim oluyor.
            # Ancak Temmuz 2022'de olduğu gibi bazen sadece 1 tablo oluyor ve o da Üretim tablosu.
            # Bu yüzden her zaman eşleşen listedeki en sonuncu [-1] tabloyu almak daha güvenlidir.
            df_gen = valid_tables[-1].copy()
            df_gen["Yıl"] = year_val
            df_gen["Ay"] = month_val
            master_generation_list.append(df_gen)
            
            if len(valid_tables) == 1:
                print(f"     [!] Bilgi: {docx_name} belgesinde sadece 1 geçerli tablo bulundu, Üretim tablosu olarak işlendi.")
        else:
            print(f"     [X] Uyarı: {docx_name} belgesinde uygun tablo bulunamadı.")
    
    
 
    def process_and_combine_generation(self):
        print("\n🚀 SADECE GENERATION (ÜRETİM) İŞLEMİ BAŞLIYOR...")
        
        master_generation_dfs = []
        years = ["2021","2022","2023","2024","2025"]
        
        for year in years:
            year_dir = os.path.join(self.raw_root, year)
            if not os.path.exists(year_dir):
                continue
                
            for file_name in os.listdir(year_dir):
                if file_name.endswith(".docx") and not file_name.startswith("~"):
                    self.run_for_file(year, file_name, master_generation_dfs)
                    
        print("\n📊 Bütün aylar tarandı. Generation (Üretim) Master dosyası oluşturuluyor...")
        
        if master_generation_dfs:
            combined_generation = pd.concat(master_generation_dfs, ignore_index=True)
            final_gen_path = os.path.join(self.final_dir, "04_Generation.xlsx")
            combined_generation.to_excel(final_gen_path, index=False)
            print(f" [✓] İŞLEM TAMAM -> Üretim Ham Master Tablosu Kaydedildi: {final_gen_path}")
        else:
            print(" [X] Hata: Birleştirilecek hiçbir Üretim tablosu bulunamadı.")


# --- ÇALIŞTIRMA ---
if __name__ == "__main__":
    BASE_PROJECT_DIR = r"C:\Users\w11\dergi2"
    
    gen_extractor = GenerationExtractor(BASE_PROJECT_DIR)
    gen_extractor.process_and_combine_generation()


🚀 SADECE GENERATION (ÜRETİM) İŞLEMİ BAŞLIYOR...
>>> Hafızaya Alınıyor (Generation): 2021/April_2021.docx -> (Yıl: 2021, Ay: April)
>>> Hafızaya Alınıyor (Generation): 2021/August_2021.docx -> (Yıl: 2021, Ay: August)
>>> Hafızaya Alınıyor (Generation): 2021/December_2021.docx -> (Yıl: 2021, Ay: December)
>>> Hafızaya Alınıyor (Generation): 2021/February_2021.docx -> (Yıl: 2021, Ay: February)
>>> Hafızaya Alınıyor (Generation): 2021/January_2021.docx -> (Yıl: 2021, Ay: January)
>>> Hafızaya Alınıyor (Generation): 2021/July_2021.docx -> (Yıl: 2021, Ay: July)
>>> Hafızaya Alınıyor (Generation): 2021/June_2021.docx -> (Yıl: 2021, Ay: June)
>>> Hafızaya Alınıyor (Generation): 2021/March_2021.docx -> (Yıl: 2021, Ay: March)
>>> Hafızaya Alınıyor (Generation): 2021/May_2021.docx -> (Yıl: 2021, Ay: May)
>>> Hafızaya Alınıyor (Generation): 2021/November_2021.docx -> (Yıl: 2021, Ay: November)
>>> Hafızaya Alınıyor (Generation): 2021/October_2021.docx -> (Yıl: 2021, Ay: October)
>>> Hafızaya Alını